In [1]:
from google.colab import drive
import os
drive.mount('/content/gdrive/')

Drive already mounted at /content/gdrive/; to attempt to forcibly remount, call drive.mount("/content/gdrive/", force_remount=True).


In [2]:
import sys
import os

# Define the path to the directory containing your package
package_parent_dir = '/content/gdrive/MyDrive/NLP_assignment/'

# Append to sys.path if it is not already present
if package_parent_dir not in sys.path:
    sys.path.append(package_parent_dir)

# Verify the path was added
print(sys.path)

['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', '/content/gdrive/MyDrive/NLP_assignment/']


In [3]:
from millionaire_client import MillionaireClient, AuthenticationError

In [ ]:
API_URL = "http://131.175.15.22:51111/"
username = ""
password = ""

In [5]:
client = MillionaireClient(API_URL)
try:
    user = client.login(username, password)
    print(f"\nWelcome, {user.username}! (Role: {user.role})")
except AuthenticationError as e:
    print(f"Login failed: {e}")


Welcome, gary! (Role: student)


In [6]:
# List available competitions
print("\n=== Available Competitions ===")
competitions = client.competitions.list_all()
for comp in competitions:
    print(f"  {comp.id}: {comp.name} ({comp.max_levels} questions)")


=== Available Competitions ===


MillionaireError: Could not connect to server at http://131.175.15.22:51111: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [7]:
# Choose a competition ID
comp_id = 1

In [8]:
def play_game(game):
  # Play the game
  while game.in_progress:
      question = game.current_question
      if not question:
          print("No question available. Game may have ended.")
          break

      print(f"\n--- Level {game.current_level} ---")
      print(f"Q: {question.text}")
      print()

      for opt in question.options:
          print(f"  [{opt.id}] {opt.text}")

      # Get time remaining
      time_left = game.time_remaining
      if time_left:
          print(f"\nTime remaining: {time_left:.1f}s")

      # Get answer
      try:
          answer_input = input("\nYour answer (option ID): ").strip()
          answer_id = int(answer_input)
      except ValueError:
          print("Invalid input. Please enter a number.")
          continue

      # Submit answer
      result = game.answer(answer_id)

      if result.correct:
          print(" CORRECT!")
          if result.game_over:
              print(f"\n CONGRATULATIONS! You completed the game!")
              print(f" Final earnings: ${result.earned_amount:,.2f}")
          else:
              print(f" Earned so far: ${result.earned_amount:,.2f}")
      elif result.timed_out:
        print("TIMED OUT!")
        print(f"\n Game Over!")
        print(f" Final earnings: ${result.earned_amount:,.2f}")
      elif not result.correct:
          print(" WRONG ANSWER!")
          print(f"\n Game Over!")
          print(f" Final earnings: ${result.earned_amount:,.2f}")

  print("\n=== Game Summary ===")
  print(f"Reached Level: {game.current_level}")
  print(f"Total Earnings: ${game.earned_amount:,.2f}")

In [9]:
import subprocess
import sys

packages = [
    'sentence-transformers',
    'rank-bm25',
    'numpy',
    'scikit-learn'
]

for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print('Dependencies installed')

Dependencies installed


In [10]:
import glob
import json
import os
import re
import urllib.request
from urllib.parse import quote_plus

import numpy as np

WIKIPEDIA_API = 'https://en.wikipedia.org/w/api.php'
LOCAL_CORPUS_DIR = None
CURATED_SOURCE_URLS = [
    {'title': 'World History Encyclopedia', 'url': 'https://www.worldhistory.org/'},
    {'title': 'Britannica Ancient History', 'url': 'https://www.britannica.com/topic/ancient-history'},
    {'title': 'Britannica Ancient Rome', 'url': 'https://www.britannica.com/place/ancient-Rome'},
    {'title': 'Britannica Ancient Greece', 'url': 'https://www.britannica.com/place/ancient-Greece'},
    {'title': 'Britannica Politics', 'url': 'https://www.britannica.com/topic/politics'},
    {'title': 'Britannica Government', 'url': 'https://www.britannica.com/topic/government'},
    {'title': 'National Archives', 'url': 'https://www.archives.gov/'},
    {'title': 'Library of Congress', 'url': 'https://www.loc.gov/'},
]


def tokenize(text: str) -> list[str]:
    return re.findall(r'[a-z0-9]+', text.lower())


def normalize_text(text: str) -> str:
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'\[\d+\]', '', text)
    return text


def fetch_wikipedia_search_results(query: str, limit: int = 5) -> list[str]:
    search_url = (
        f'{WIKIPEDIA_API}?action=query&list=search&srsearch={quote_plus(query)}'
        f'&srlimit={limit}&format=json&utf8=1&redirects=1'
    )
    with urllib.request.urlopen(urllib.request.Request(search_url, headers={'User-Agent': 'Mozilla/5.0'})) as response:
        data = json.load(response)
    return [item['title'] for item in data.get('query', {}).get('search', [])]


def fetch_wikipedia_page(title: str):
    content_url = (
        f'{WIKIPEDIA_API}?action=query&prop=extracts&explaintext=true'
        f'&titles={quote_plus(title)}&format=json&utf8=1&redirects=1'
    )
    with urllib.request.urlopen(urllib.request.Request(content_url, headers={'User-Agent': 'Mozilla/5.0'})) as response:
        data = json.load(response)

    pages = data.get('query', {}).get('pages', {})
    page = next(iter(pages.values()), {})
    text = page.get('extract', '')
    resolved_title = page.get('title', title)
    if not text:
        return None
    return {
        'source': 'wikipedia',
        'title': resolved_title,
        'text': text,
    }


def load_local_documents(folder: str | None):
    documents = []
    if not folder or not os.path.isdir(folder):
        return documents

    patterns = [os.path.join(folder, '**', '*.txt'), os.path.join(folder, '**', '*.md')]
    for pattern in patterns:
        for path in sorted(glob.glob(pattern, recursive=True)):
            try:
                with open(path, 'r', encoding='utf-8') as handle:
                    text = handle.read().strip()
                if text:
                    documents.append({'source': 'local', 'title': os.path.basename(path), 'path': path, 'text': text})
            except Exception:
                continue
    return documents


def fetch_url_document(url: str, title: str = ''):
    try:
        with urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})) as response:
            raw_text = response.read().decode('utf-8', errors='ignore')
    except Exception:
        return None

    text = normalize_text(raw_text)
    if len(text) < 200:
        return None

    return {
        'source': 'web',
        'title': title or url,
        'url': url,
        'text': text,
    }


def collect_documents(question: str, option_texts: list[str], max_pages: int = 8):
    documents = []
    seen_titles = set()

    documents.extend(load_local_documents(LOCAL_CORPUS_DIR))

    search_queries = [question]
    search_queries.extend(option_texts)
    search_queries.extend([f'{question} {option}' for option in option_texts])

    for query in search_queries:
        try:
            titles = fetch_wikipedia_search_results(query, limit=3)
        except Exception:
            continue

        for title in titles:
            normalized_title = title.lower()
            if normalized_title in seen_titles:
                continue
            seen_titles.add(normalized_title)

            try:
                document = fetch_wikipedia_page(title)
            except Exception:
                continue

            if document and len(document['text']) > 200:
                documents.append(document)

            if len(documents) >= max_pages:
                return documents

    for item in CURATED_SOURCE_URLS:
        url = item.get('url') if isinstance(item, dict) else item
        title = item.get('title', '') if isinstance(item, dict) else ''
        if not url:
            continue

        document = fetch_url_document(url, title=title)
        if document:
            documents.append(document)
        if len(documents) >= max_pages:
            return documents

    return documents


print('Document utilities ready')

Document utilities ready


In [11]:
def clean_text(text: str) -> str:
    text = text.replace('\xa0', ' ')
    text = re.sub(r'<ref[^>]*>.*?</ref>', ' ', text, flags=re.IGNORECASE | re.DOTALL)
    text = re.sub(r'<ref[^/]*/>', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\{\{.*?\}\}', ' ', text, flags=re.DOTALL)
    text = re.sub(r'\[\[File:.*?\]\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[\[(?:[^\]|]+\|)?([^\]]+)\]\]', r'\1', text)
    text = re.sub(r"''+", '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def split_into_chunks(text: str, chunk_size: int = 1200, overlap: int = 180):
    text = clean_text(text)
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks = []
    current_sentences = []
    current_length = 0

    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue

        sentence_length = len(sentence)
        if current_sentences and current_length + sentence_length + 1 > chunk_size:
            chunk = ' '.join(current_sentences).strip()
            if chunk:
                chunks.append(chunk)

            if overlap > 0 and current_sentences:
                overlap_sentences = []
                overlap_length = 0
                for previous_sentence in reversed(current_sentences):
                    overlap_sentences.insert(0, previous_sentence)
                    overlap_length += len(previous_sentence) + 1
                    if overlap_length >= overlap:
                        break
                current_sentences = overlap_sentences[:]
                current_length = sum(len(item) + 1 for item in current_sentences)
            else:
                current_sentences = []
                current_length = 0

        current_sentences.append(sentence)
        current_length += sentence_length + 1

    if current_sentences:
        chunk = ' '.join(current_sentences).strip()
        if chunk:
            chunks.append(chunk)

    return chunks


def chunk_documents(documents: list[dict], chunk_size: int = 1200, overlap: int = 180):
    chunks = []
    for document_index, document in enumerate(documents):
        for chunk_index, chunk in enumerate(split_into_chunks(document['text'], chunk_size=chunk_size, overlap=overlap)):
            chunks.append({
                'doc_index': document_index,
                'chunk_index': chunk_index,
                'source': document.get('source', 'unknown'),
                'title': document.get('title', ''),
                'path': document.get('path'),
                'text': chunk,
            })
    return chunks


print('Cleaning and chunking utilities ready')

Cleaning and chunking utilities ready


In [12]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


EMBEDDING_MODEL_NAME = 'all-mpnet-base-v2'
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(f'Loaded embedding model: {EMBEDDING_MODEL_NAME}')


class HybridRetriever:
    def __init__(self, chunks: list[dict], model=None):
        self.chunks = chunks
        self.chunk_texts = [chunk['text'] for chunk in chunks]
        self.tokenized_chunks = [tokenize(text) for text in self.chunk_texts]
        self.bm25 = BM25Okapi(self.tokenized_chunks) if self.tokenized_chunks else None
        self.model = model or embedding_model
        self.chunk_embeddings = self.model.encode(self.chunk_texts, convert_to_numpy=True, show_progress_bar=False) if self.chunk_texts else np.empty((0, 768))

    @staticmethod
    def _normalize_scores(scores):
        scores = np.asarray(scores, dtype=float)
        if scores.size == 0:
            return scores
        min_score = scores.min()
        max_score = scores.max()
        if np.isclose(max_score, min_score):
            return np.zeros_like(scores)
        return (scores - min_score) / (max_score - min_score)

    def retrieve(self, query: str, top_k: int = 8, bm25_weight: float = 0.45):
        if not self.chunks:
            return []

        query_tokens = tokenize(query)
        bm25_scores = self.bm25.get_scores(query_tokens) if self.bm25 is not None else np.zeros(len(self.chunks))
        query_embedding = self.model.encode(query, convert_to_numpy=True, show_progress_bar=False)
        dense_scores = cosine_similarity([query_embedding], self.chunk_embeddings)[0] if len(self.chunk_embeddings) else np.zeros(len(self.chunks))

        bm25_scores = self._normalize_scores(bm25_scores)
        dense_scores = self._normalize_scores(dense_scores)
        final_scores = bm25_weight * bm25_scores + (1.0 - bm25_weight) * dense_scores

        top_indices = np.argsort(final_scores)[::-1][:top_k]
        return [
            {
                'score': float(final_scores[index]),
                'bm25_score': float(bm25_scores[index]),
                'dense_score': float(dense_scores[index]),
                'chunk': self.chunks[index],
            }
            for index in top_indices
        ]


print('Hybrid retriever ready')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded embedding model: all-mpnet-base-v2
Hybrid retriever ready


In [13]:
def score_option_against_chunks(option_text: str, retrieved_chunks: list[dict]) -> float:
    if not retrieved_chunks:
        return 0.0

    option_tokens = set(tokenize(option_text))
    combined_text = ' '.join(item['chunk']['text'] for item in retrieved_chunks)
    combined_tokens = set(tokenize(combined_text))
    overlap = len(option_tokens & combined_tokens) / max(len(option_tokens), 1)
    exact_phrase = 1.0 if normalize_text(option_text).lower() in normalize_text(combined_text).lower() else 0.0
    mean_retrieval_score = float(np.mean([item['score'] for item in retrieved_chunks[:3]])) if retrieved_chunks else 0.0
    return mean_retrieval_score + (0.30 * overlap) + (0.25 * exact_phrase)


def rank_options(question_text: str, options, retriever: HybridRetriever, top_k: int = 8):
    ranked_options = []

    for option in options:
        option_text = option.text if hasattr(option, 'text') else option['text']
        option_id = option.id if hasattr(option, 'id') else option['id']
        query = f'{question_text} {option_text}'
        retrieved_chunks = retriever.retrieve(query, top_k=top_k)
        score = score_option_against_chunks(option_text, retrieved_chunks)
        ranked_options.append({
            'option_id': option_id,
            'option_text': option_text,
            'score': float(score),
            'retrieved_chunks': retrieved_chunks,
        })

    ranked_options.sort(key=lambda item: item['score'], reverse=True)
    return ranked_options


print('Option scorer ready')

Option scorer ready


In [14]:
def answer_question_rag(question_text: str, options, llama_model, llama_tokenizer, max_pages: int = 8, top_k: int = 8):
    """Answer question using RAG for context + Llama 3.2 for reasoning."""
    option_texts = [option.text if hasattr(option, 'text') else option['text'] for option in options]
    documents = collect_documents(question_text, option_texts, max_pages=max_pages)

    if not documents:
        answer_id = options[0].id if hasattr(options[0], 'id') else options[0]['id']
        answer_text = options[0].text if hasattr(options[0], 'text') else options[0]['text']
        return {
            'answer_id': answer_id,
            'answer_text': answer_text,
            'confidence': 0.1,
            'scores': [],
            'documents': [],
            'chunks': [],
        }

    chunks = chunk_documents(documents)
    context_text = "\n".join([chunk['text'][:200] for chunk in chunks[:5]])
    
    prompt = f"""Based on the following context, which option is correct?

Question: {question_text}

Context:
{context_text}

Options:
"""
    for i, opt in enumerate(options):
        opt_text = opt.text if hasattr(opt, 'text') else opt['text']
        prompt += f"\n{chr(65+i)}. {opt_text}"
    
    prompt += "\n\nAnswer (A/B/C/D): "
    
    inputs = llama_tokenizer(prompt, return_tensors="pt").to(llama_model.device)
    outputs = llama_model.generate(**inputs, max_new_tokens=1, temperature=0.1)
    answer_letter = llama_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()[-1].upper()
    
    option_idx = ord(answer_letter) - 65 if answer_letter in 'ABCD' else 0
    option_idx = min(option_idx, len(options) - 1)
    
    best = options[option_idx]
    answer_id = best.id if hasattr(best, 'id') else best['id']
    answer_text = best.text if hasattr(best, 'text') else best['text']
    
    return {
        'answer_id': answer_id,
        'answer_text': answer_text,
        'confidence': 0.8,
        'scores': [],
        'documents': documents,
        'chunks': chunks,
    }

print('RAG pipeline with Llama ready')

RAG pipeline with Llama ready


In [26]:
# Prepare Ancient History training data for fine-tuning
print("Preparing Ancient History training dataset...")

# Sample training data (Q&A pairs on Ancient History & Politics)
training_data = [
    {
        "question": "What term describes the geographical regions most directly influenced by Greek and Roman culture?",
        "answer": "The Greco-Roman World"
    },
    {
        "question": "What period of Egyptian history did the Nineteenth Dynasty rule?",
        "answer": "New Kingdom"
    },
    {
        "question": "Who was the first Roman Emperor?",
        "answer": "Augustus"
    },
    {
        "question": "What was the primary purpose of the Roman aqueducts?",
        "answer": "To transport water"
    },
    {
        "question": "Which ancient Greek city-state was known for its military strength?",
        "answer": "Sparta"
    },
    {
        "question": "What was the primary governing body of ancient Rome?",
        "answer": "The Senate"
    },
    {
        "question": "Who was the founder of the Persian Empire?",
        "answer": "Cyrus the Great"
    },
    {
        "question": "What was the primary religion of ancient Egypt?",
        "answer": "Polytheism"
    },
    {
        "question": "Which ancient wonder was located in Alexandria?",
        "answer": "The Lighthouse of Alexandria"
    },
    {
        "question": "What was the primary focus of Stoic philosophy?",
        "answer": "Virtue and duty"
    },
    {
        "question": "Who conquered the Persian Empire?",
        "answer": "Alexander the Great"
    },
    {
        "question": "What system of government did ancient Athens use?",
        "answer": "Democracy"
    },
    {
        "question": "Which Roman general crossed the Rubicon?",
        "answer": "Julius Caesar"
    },
    {
        "question": "What was the main trade route connecting East and West?",
        "answer": "The Silk Road"
    },
    {
        "question": "Who was the greatest military commander of ancient Carthage?",
        "answer": "Hannibal"
    }
]

print(f"✓ Loaded {len(training_data)} training examples")

Preparing Ancient History training dataset...
✓ Loaded 15 training examples


In [31]:
# Setup LoRA fine-tuning for Llama
print("Setting up LoRA fine-tuning...")

from peft import LoraConfig, get_peft_model, TaskType
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import Dataset

# Set padding token for tokenizer
if llama_tokenizer.pad_token is None:
    llama_tokenizer.pad_token = llama_tokenizer.eos_token
    print(f"✓ Padding token set to: {llama_tokenizer.pad_token}")

# Create instruction-following prompts
def create_prompt(example):
    return f"""Below is an instruction with input. Write a response that appropriately completes the request.

### Instruction:
Answer the following ancient history question:

### Input:
{example['question']}

### Response:
{example['answer']}"""

# Prepare dataset
prompts = [create_prompt(ex) for ex in training_data]
dataset = Dataset.from_dict({'text': prompts})

# Split into train/eval (80/20)
split_dataset = dataset.train_test_split(test_size=0.2)
train_dataset = split_dataset['train']
eval_dataset = split_dataset['test']

print(f"✓ Training set: {len(train_dataset)} examples")
print(f"✓ Eval set: {len(eval_dataset)} examples")

# LoRA configuration for efficient fine-tuning
lora_config = LoraConfig(
    r=16,  # LoRA rank
    lora_alpha=32,  # LoRA scaling
    target_modules=['q_proj', 'v_proj'],  # Attention layers to adapt
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM
)

# Apply LoRA to the model
print("Applying LoRA to Llama model...")
llama_model = get_peft_model(llama_model, lora_config)
llama_model.print_trainable_parameters()

print("✓ LoRA configured")

Setting up LoRA fine-tuning...
✓ Padding token set to: <|eot_id|>
✓ Training set: 12 examples
✓ Eval set: 3 examples
Applying LoRA to Llama model...
trainable params: 1,703,936 || all params: 1,237,518,336 || trainable%: 0.1377
✓ LoRA configured


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [32]:
# Fine-tune Llama on Ancient History data
print("Starting fine-tuning on T4 GPU...")

# Tokenize dataset
def tokenize_function(examples):
    return llama_tokenizer(examples['text'], truncation=True, max_length=512)

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=['text'])
tokenized_eval = eval_dataset.map(tokenize_function, batched=True, remove_columns=['text'])

# Training configuration optimized for T4
training_args = TrainingArguments(
    output_dir='/content/llama-finetuned',
    num_train_epochs=3,
    per_device_train_batch_size=2,  # Small batch for T4
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,  # Effective batch = 2 * 4 = 8
    warmup_steps=50,
    weight_decay=0.01,
    logging_steps=5,
    eval_strategy='steps',
    eval_steps=10,
    save_strategy='steps',
    save_steps=10,
    learning_rate=5e-4,
    fp16=True,  # Mixed precision for T4
    push_to_hub=False,
    report_to=[],  # Disable reporting
)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    llama_tokenizer,
    mlm=False  # Causal LM, not masked LM
)

# Trainer
trainer = Trainer(
    model=llama_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator
)

# Train
trainer.train()

print("✓ Fine-tuning complete!")
print(f"Model saved to {training_args.output_dir}")

Starting fine-tuning on T4 GPU...


Map:   0%|          | 0/12 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss


✓ Fine-tuning complete!
Model saved to /content/llama-finetuned


In [29]:
# Updated answer function using fine-tuned Llama
def answer_question_rag_finetuned(question_text: str, options, llama_model, llama_tokenizer, max_pages: int = 8, top_k: int = 8):
    """Answer using RAG + fine-tuned Llama reasoning."""
    option_texts = [option.text if hasattr(option, 'text') else option['text'] for option in options]
    documents = collect_documents(question_text, option_texts, max_pages=max_pages)

    if not documents:
        answer_id = options[0].id if hasattr(options[0], 'id') else options[0]['id']
        answer_text = options[0].text if hasattr(options[0], 'text') else options[0]['text']
        return {
            'answer_id': answer_id,
            'answer_text': answer_text,
            'confidence': 0.1,
            'scores': [],
            'documents': [],
            'chunks': [],
        }

    chunks = chunk_documents(documents)
    context_text = "\n".join([chunk['text'][:200] for chunk in chunks[:5]])
    
    prompt = f"""Below is an ancient history question with context. Choose the correct answer.

Question: {question_text}

Context: {context_text}

Options:
"""
    for i, opt in enumerate(options):
        opt_text = opt.text if hasattr(opt, 'text') else opt['text']
        prompt += f"\n{chr(65+i)}. {opt_text}"
    
    prompt += "\n\nCorrect answer (A/B/C/D): "
    
    inputs = llama_tokenizer(prompt, return_tensors="pt").to(llama_model.device)
    outputs = llama_model.generate(**inputs, max_new_tokens=1, temperature=0.1, top_p=0.9)
    answer_letter = llama_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()[-1].upper()
    
    option_idx = ord(answer_letter) - 65 if answer_letter in 'ABCD' else 0
    option_idx = min(option_idx, len(options) - 1)
    
    best = options[option_idx]
    answer_id = best.id if hasattr(best, 'id') else best['id']
    answer_text = best.text if hasattr(best, 'text') else best['text']
    
    return {
        'answer_id': answer_id,
        'answer_text': answer_text,
        'confidence': 0.85,
        'scores': [],
        'documents': documents,
        'chunks': chunks,
    }

print('Fine-tuned RAG pipeline ready')

Fine-tuned RAG pipeline ready


In [ ]:
# Install Llama dependencies
import subprocess
import sys

print('Installing model dependencies...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers', 'bitsandbytes', 'peft', 'accelerate'])
print('✓ Dependencies installed')

# Load Llama 3.2 with Hugging Face authentication
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

print('\nLoading Llama 3.2 1B with 4-bit quantization...')

# HuggingFace token
hf_token = ''

# 4-bit quantization config for memory efficiency on T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load Llama 3.2 1B (faster than 8B, fits on T4)
model_name = 'meta-llama/Llama-3.2-1B-Instruct'
llama_tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token, trust_remote_code=True)
llama_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map='auto',
    token=hf_token,
    trust_remote_code=True
)

print(f'✓ Llama 3.2 1B loaded on {llama_model.device}')
print('Ready for game! Each question will use Llama to reason about the answer.')

Installing model dependencies...
✓ Dependencies installed

Loading Llama 3.2 1B with 4-bit quantization...


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

✓ Llama 3.2 1B loaded on cuda:0
Ready for game! Each question will use Llama to reason about the answer.


In [34]:
def play_game_with_finetuned_rag(game, llama_model, llama_tokenizer, max_pages: int = 8, top_k: int = 8):
    """Play game using fine-tuned Llama + RAG"""
    round_num = 0
    correct_count = 0

    while game.in_progress:
        round_num += 1
        question = game.current_question
        if not question:
            print('No question available. Game may have ended.')
            break

        print(f"\n{'=' * 72}")
        print(f'Round {round_num} | Level {game.current_level}')
        print(f"{'=' * 72}")
        print(f'Q: {question.text}\n')

        for option in question.options:
            print(f'  [{option.id}] {option.text}')

        # Use fine-tuned model for better Ancient History accuracy
        rag_result = answer_question_rag_finetuned(question.text, question.options, llama_model, llama_tokenizer, max_pages=max_pages, top_k=top_k)
        answer_id = rag_result['answer_id']
        answer_text = rag_result['answer_text']
        confidence = rag_result['confidence']

        print(f"\nFine-tuned Llama answer: [{answer_id}] {answer_text}")
        print(f'Confidence: {confidence:.3f}')

        result = game.answer(answer_id)

        if result.correct:
            print('✓ CORRECT!')
            correct_count += 1
            if result.game_over:
                print('\nCONGRATULATIONS! You completed the game!')
                print(f'Final earnings: ${result.earned_amount:,.2f}')
                break
            print(f'Earned so far: ${result.earned_amount:,.2f}')
        elif result.timed_out:
            print('⏱ TIMED OUT!')
            print(f'Final earnings: ${result.earned_amount:,.2f}')
            break
        else:
            print('✗ WRONG ANSWER!')
            print(f'Final earnings: ${result.earned_amount:,.2f}')
            break

    print('\n=== Game Summary ===')
    print(f'Rounds played: {round_num}')
    print(f'Correct: {correct_count}/{round_num}')
    if round_num > 0:
        print(f'Accuracy: {100 * correct_count / round_num:.1f}%')
    print(f'Final Level: {game.current_level}')
    print(f'Total Earnings: ${game.earned_amount:,.2f}')


print('Fine-tuned game function ready')

Fine-tuned game function ready


In [18]:
def play_game_with_rag(game, llama_model, llama_tokenizer, max_pages: int = 8, top_k: int = 8):
    round_num = 0
    correct_count = 0

    while game.in_progress:
        round_num += 1
        question = game.current_question
        if not question:
            print('No question available. Game may have ended.')
            break

        print(f"\n{'=' * 72}")
        print(f'Round {round_num} | Level {game.current_level}')
        print(f"{'=' * 72}")
        print(f'Q: {question.text}\n')

        for option in question.options:
            print(f'  [{option.id}] {option.text}')

        rag_result = answer_question_rag(question.text, question.options, llama_model, llama_tokenizer, max_pages=max_pages, top_k=top_k)
        answer_id = rag_result['answer_id']
        answer_text = rag_result['answer_text']
        confidence = rag_result['confidence']

        print(f"\nLlama answer: [{answer_id}] {answer_text}")
        print(f'Confidence: {confidence:.3f}')

        result = game.answer(answer_id)

        if result.correct:
            print('CORRECT!')
            correct_count += 1
            if result.game_over:
                print('\nCONGRATULATIONS! You completed the game!')
                print(f'Final earnings: ${result.earned_amount:,.2f}')
                break
            print(f'Earned so far: ${result.earned_amount:,.2f}')
        elif result.timed_out:
            print('TIMED OUT!')
            print(f'Final earnings: ${result.earned_amount:,.2f}')
            break
        else:
            print('WRONG ANSWER!')
            print(f'Final earnings: ${result.earned_amount:,.2f}')
            break

    print('\n=== Game Summary ===')
    print(f'Rounds played: {round_num}')
    print(f'Correct: {correct_count}/{round_num}')
    print(f'Accuracy: {100 * correct_count / round_num:.1f}%')
    print(f'Final Level: {game.current_level}')
    print(f'Total Earnings: ${game.earned_amount:,.2f}')


print('Game integration ready')

Game integration ready


In [41]:
# Start game with fine-tuned Llama RAG
print('Starting game with fine-tuned Llama 3.2 RAG...')
game = client.game.start(competition_id=comp_id)
print(f'Session ID: {game.session_id}\n')

# Play with fine-tuned model for better Ancient History answers
play_game_with_finetuned_rag(game, llama_model, llama_tokenizer)

Starting game with fine-tuned Llama 3.2 RAG...
Session ID: 36489


Round 1 | Level 1
Q: Which architectural order is recognized by its voluted capital, featuring acanthus leaves and volutes similar to those of a ram's horn?

  [0] Corinthian order
  [1] Hellenistic order
  [2] Doric order
  [3] Ionic order


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



Fine-tuned Llama answer: [0] Corinthian order
Confidence: 0.850
✓ CORRECT!
Earned so far: $100.00

Round 2 | Level 2
Q: According to Hittite documents, which term is believed to be related to the name of the Achaeans, and is mentioned in the Tawagalawa letter?

  [0] Tawagalawa
  [1] Wilusa
  [2] Madduwatta
  [3] Ahhiyawa


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



Fine-tuned Llama answer: [0] Tawagalawa
Confidence: 0.850
✗ WRONG ANSWER!
Final earnings: $100.00

=== Game Summary ===
Rounds played: 2
Correct: 1/2
Accuracy: 50.0%
Final Level: 2
Total Earnings: $100.00


In [24]:
# Show leaderboard position
lb = client.leaderboard.get(competition_id=comp_id, limit=10)
print(f"\n=== Leaderboard for {lb.competition.name} ===")
for i, entry in enumerate(lb.entries[:20], 1):
    marker = " <-- YOU" if entry.username == username else ""
    print(f"  {i}. {entry.username}: ${entry.score:,.2f} (Level {entry.reached_level}){marker}")


=== Leaderboard for Ancient History and Politics ===
  1. AleAssini: $1,024,000.00 (Level 15)
  2. supreme_leader: $1,024,000.00 (Level 15)
  3. luca_bordin: $1,024,000.00 (Level 15)
  4. Anonymous: $1,024,000.00 (Level 15)
  5. Jasmin: $1,024,000.00 (Level 15)
  6. MatteoVitali: $1,024,000.00 (Level 15)
  7. TheLastGuessbender: $1,024,000.00 (Level 15)
  8. gab: $1,024,000.00 (Level 15)
  9. Zero37: $1,024,000.00 (Level 15)
  10. kristiduro: $1,024,000.00 (Level 15)
